In [1]:
import numpy as np, pandas as pd, re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report


In [5]:
df = pd.read_csv("sentiment_data.csv")


stopwords = {"the","a","an","is","and","or","was","it","this","that","of","to","very","but","be","in","on","for","so","at","as","are","were","from"}

def preprocess(t):
    t = t.lower()
    t = re.sub(r"[^a-z\s]", "", t)
    return [w for w in t.split() if w not in stopwords]



In [6]:

vocab = Counter()
for t in df["tokens"]: vocab.update(t)
vocab = {w:i for i,(w,_) in enumerate(vocab.most_common(5000))}


KeyError: 'tokens'

In [4]:

def vectorize(t):
    v = np.zeros(len(vocab))
    for w in t:
        if w in vocab: v[vocab[w]] += 1
    return v

X = np.array([vectorize(t) for t in df["tokens"]])
y = df["Sentiment"].values

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
model = MultinomialNB()
model.fit(X_train,y_train)
y_pred = model.predict(X_test)


MemoryError: Unable to allocate 8.98 GiB for an array with shape (240928, 5000) and data type float64

In [ ]:

print("Accuracy:",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

def predict_sentiment(s):
    t = preprocess(s)
    v = vectorize(t).reshape(1,-1)
    p = model.predict(v)[0]
    return {0:"Negative",1:"Neutral",2:"Positive"}[p]

print(predict_sentiment("I love using Apple Pay, it's fast and easy"))
print(predict_sentiment("Apple Pay never works when I need it"))
print(predict_sentiment("It’s okay, not too bad but not great either"))

In [1]:

def extractive_summarization(text, summary_length):
    sentences = text.split('. ')
    sentence_scores = {}
    for sentence in sentences:
        score = 0
        words = sentence.split()
        for word in words:
            score += words.count(word)
        sentence_scores[sentence] = score
    top_sentences = sorted(sentence_scores.items(), key=lambda x: x[1], reverse=True)[:summary_length]
    summary = '. '.join([sentence for sentence, score in top_sentences])
    return summary

# Example usage
text = "This is a sample text. The text is used for summarization. Summarization is a technique used in NLP."
summary_length = 2
print(extractive_summarization(text, summary_length))




def abstractive_summarization(text, summary_length):
    words = text.split()
    word_freq = {}
    for word in words:
        if word not in word_freq:
            word_freq[word] = 1
        else:
            word_freq[word] += 1
    top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:summary_length]
    summary = ' '.join([word for word, freq in top_words])
    return summary

# Example usage
text = "This is a sample text. The text is used for summarization. Summarization is a technique used in NLP."
summary_length = 5
print(abstractive_summarization(text, summary_length))



# with open('document.txt', 'r') as f:
#     text = f.read()
# summary = extractive_summarization(text, 5)
# print(summary)


Summarization is a technique used in NLP.. The text is used for summarization
is a used This sample
